# 🛶 jangada — Quickstart no Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nerigleston/jangada-docs/blob/main/examples/notebooks/jangada_quickstart.ipynb)

Uma camada fina sobre Anthropic, OpenAI, Groq e Gemini: **troque o provider sem mudar o código**, com structured output, vision, RAG, tools, agentes, guardrails, MCP, retry e fallback.

Este notebook roda do zero: instala, pede suas chaves com segurança e executa os casos principais.

- PyPI: https://pypi.org/project/jangada-ai/
- Docs: https://docs.jangada.dev.br
- GitHub (docs): https://github.com/nerigleston/jangada-docs

## 1. Instalar

In [ ]:
%pip install -q "jangada-ai[all]"

## 2. Suas chaves de API

Rode a célula e cole a(s) chave(s) que você tem. As outras você pode deixar em branco — a jangada só usa o provider que você chamar. As chaves ficam só na sessão (não são salvas).

In [ ]:
import os
from getpass import getpass

for var in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GEMINI_API_KEY"]:
    if not os.environ.get(var):
        valor = getpass(f"{var} (Enter para pular): ")
        if valor:
            os.environ[var] = valor
print("Chaves configuradas:", [v for v in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GEMINI_API_KEY"] if os.environ.get(v)])

## 3. Primeira chamada

Troque `(provider, modelo)` por um que você tenha chave.

In [ ]:
from jangada_ai import LLM

llm = LLM("openai", "gpt-4o-mini")  # ou ("groq", "llama-3.3-70b-versatile"), ("gemini", "gemini-2.5-flash"), ("anthropic", "claude-haiku-4-5-20251001")
comp = llm.complete("Explique o que é uma jangada em uma frase.")
print(comp.text)
print("tokens:", comp.usage, "| custo US$:", comp.cost)

## 4. O pitch: trocar de provider sem mudar o código

A mesma chamada vale para os quatro — só muda o par `(provider, modelo)`.

In [ ]:
candidatos = [
    ("openai", "gpt-4o-mini"),
    ("groq", "llama-3.3-70b-versatile"),
    ("gemini", "gemini-2.5-flash"),
    ("anthropic", "claude-haiku-4-5-20251001"),
]
chave = {"openai": "OPENAI_API_KEY", "groq": "GROQ_API_KEY", "gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}

for provider, modelo in candidatos:
    if not os.environ.get(chave[provider]):
        continue
    texto = LLM(provider, modelo).complete("Diga 'olá' em uma palavra.").text
    print(f"{provider:10} {modelo:35} -> {texto.strip()}")

## 5. Structured output (texto → Pydantic)

Uma chamada `parse()` devolve um objeto validado — em qualquer provider.

In [ ]:
from pydantic import BaseModel

class Pessoa(BaseModel):
    nome: str
    idade: int
    cidade: str | None = None

p = llm.parse("Extraia: a Ana tem 30 anos e mora em Recife.", Pessoa).parsed
print(p)

## 6. Vision: extrair dados de uma imagem

Faça upload de uma foto (ex.: uma nota fiscal) e extraia structured + vision juntos. Use um modelo com visão (gpt-4o-mini, gemini-2.5-flash, claude-haiku-4-5).

In [ ]:
from google.colab import files
from pydantic import BaseModel

class Item(BaseModel):
    descricao: str
    valor_total: float

class Nota(BaseModel):
    estabelecimento: str
    itens: list[Item]
    valor_a_pagar: float

enviados = files.upload()  # selecione uma imagem
caminho = next(iter(enviados))
nota = LLM("openai", "gpt-4o-mini").parse("Extraia os dados desta nota fiscal. Não invente.", Nota, images=[caminho]).parsed
print(nota.estabelecimento, "-> R$", nota.valor_a_pagar)
for i in nota.itens:
    print(" ", i.descricao, i.valor_total)

## 7. RAG em ~10 linhas (busca híbrida: BM25 + vetorial)

Responde com base nos seus textos. Em memória aqui; troque por `vector_store('postgresql://...')` em produção. (Precisa de chave OpenAI para o embedding.)

In [ ]:
from jangada_ai.rag import RAG, InMemoryVectorStore

embedder = LLM("openai", "text-embedding-3-small")
chat = LLM("openai", "gpt-4o-mini")
rag = RAG(embedder, InMemoryVectorStore(), chat=chat, k=3, alpha=0.5)
rag.add_texts([
    "A jangada troca de provider mudando só LLM('provider', 'modelo').",
    "O fallback é acionado em rate limit, 5xx ou timeout.",
    "A busca híbrida combina BM25 (palavras) e vetorial (semântica) por RRF.",
])
resp = rag.ask("Quando o fallback é acionado?")
print(resp.text)
print("fontes:", len(resp.sources))

## 8. Resiliência: retry + fallback + custo

Defina um reserva: se o primário falhar (rate limit, 5xx, timeout), cai pro outro automaticamente. (Precisa de 2 providers.)

In [ ]:
primario = LLM("openai", "gpt-4o-mini")
reserva = LLM("anthropic", "claude-haiku-4-5-20251001")
robusto = primario.with_fallback(reserva)

comp = robusto.complete("Resuma 'tolerância a falhas' em uma frase.")
print(comp.text)
print("usou:", comp.provider, comp.model, "| custo US$:", comp.cost)

## 9. Agentes: um agente que usa ferramentas sozinho

Um `Agent` recebe um papel + ferramentas (funções Python) e roda o loop de tool calling sozinho — decide quando chamar cada ferramenta até concluir a tarefa. Ótimo para forçar contas exatas (sem alucinação) e buscar dados.

In [ ]:
from jangada_ai import Agent

def somar(valores: list[float]) -> float:
    """Soma uma lista de números e devolve o total. Use SEMPRE para contas."""
    return sum(valores)

def cidade_da_pessoa(nome: str) -> str:
    """Retorna a cidade onde uma pessoa mora (base fictícia)."""
    return {"Ana": "Recife", "Bruno": "Curitiba"}.get(nome, "desconhecida")

agente = Agent(
    LLM("openai", "gpt-4o-mini"),
    role="assistente de dados",
    goal="responder com precisão usando as ferramentas disponíveis",
    tools=[somar, cidade_da_pessoa],
)
res = agente.run("Quanto é 12.5 + 7.5 + 80? E em que cidade a Ana mora?")
print(res.text)
print("iterações:", res.iterations, "| custo US$:", res.cost)

## 10. Guardrails: mantenha a LLM dentro do escopo

`ScopeGuard` mantém o modelo num domínio e **barra falas fora dele** — com uma blocklist (regex, custo zero) e um *judge* LLM barato que classifica o escopo. Ao barrar, a chamada curto-circuita com uma `Completion` de recusa (sem custo). Dica: use um judge **capaz** (ex.: `gpt-4o-mini`); modelos muito fracos deixam passar.

In [ ]:
from jangada_ai import LLM, ScopeGuard

# Domínio permitido + judge barato que classifica o escopo.
guard = ScopeGuard(
    "Suporte de um app de finanças pessoais: contas, planos, pagamentos e uso do app. "
    "NÃO responde outros assuntos (receitas, esportes, política, programação genérica).",
    judge=LLM("openai", "gpt-4o-mini"),
    message="Desculpe, só posso ajudar com o suporte do app. 🙂",
)
suporte = LLM("openai", "gpt-4o-mini", guardrails=[guard])

for pergunta in ["Como cancelo meu plano?", "Quanto foi o jogo do Flamengo?"]:
    r = suporte.complete(pergunta)
    barrado = bool(r.raw and r.raw.get("guardrail"))
    print(f"[{'BARRADO' if barrado else 'OK'}] {pergunta}\n  -> {r.text}\n")

## 11. Squad: um time de agentes (handoff)

`Squad` orquestra vários `Agent` em sequência: a saída de um vira o contexto do próximo (handoff). Cada agente tem seu papel — aqui um pesquisador levanta os pontos e um redator transforma num texto final. Agrega `usage`/`cost` de todos.

In [ ]:
from jangada_ai import Agent, Squad

pesquisador = Agent(
    LLM("openai", "gpt-4o-mini"),
    role="pesquisador",
    goal="levantar 3 pontos objetivos sobre o tema",
)
redator = Agent(
    LLM("openai", "gpt-4o-mini"),
    role="redator",
    goal="transformar os pontos num parágrafo claro e conciso",
)

time = Squad([pesquisador, redator])  # sequencial: pesquisador -> redator
out = time.run("Benefícios de trocar de provider de LLM sem mudar o código.")
print(out.text)
print("\ncusto total US$:", out.cost)

## 12. MCP: um agente que usa ferramentas de um servidor externo

Com `MCPClient` + `run_agent`, o agente conecta a um servidor **MCP** (Model Context Protocol), lista as ferramentas expostas e roda o loop sozinho — funciona em qualquer provider. É async; no Colab use `await` direto. Precisa do extra `mcp` (já incluso no `[all]`) e da URL de um servidor MCP.

In [ ]:
from jangada_ai import LLM, MCPClient, run_agent

# Troque pela URL de um servidor MCP (streamable-http). Ex. público de teste:
MCP_URL = "https://remote.mcpservers.org/fetch/mcp"

agente = LLM("openai", "gpt-4o-mini")  # qualquer provider serve
async with MCPClient(MCP_URL) as mcp:
    resposta = await run_agent(
        agente,
        "Liste as ferramentas disponíveis e use uma delas para buscar https://example.com",
        client=mcp,
    )
print(resposta.text)

## Próximos passos

- **Cookbook** (receitas completas): https://github.com/nerigleston/jangada-docs/tree/main/examples/cookbook
- **Tutoriais**: https://github.com/nerigleston/jangada-docs/tree/main/docs
- **Agente MCP, transcrição de áudio, detecção de objetos** e mais nos docs.

Feito com 🛶 jangada.